# 09 · Ensemble, Calibration & Test Evaluation

**Purpose.** Combine the best models, calibrate them, and score them once on the untouched test set: confidence intervals, per class, per source, per skin type, grouped taxonomy.

| | |
|---|---|
| **Reads** | `best_models.json`, `data/splits/*` |
| **Writes** | `reports/results/test_results.json`, `test_predictions.csv`, figures |
| **Run time (Colab T4)** | ~10 min |
| **Next notebook** | 10 · Open-Set Evaluation |

Run the cells from top to bottom. All shared logic lives in `src/dermalens/`; this
notebook only calls it and explains each decision.

## 0. Setup

The same cell starts every notebook. It (1) connects Google Drive on Colab,
(2) installs missing libraries on Colab, (3) makes `src/` importable,
(4) finds every project folder and (5) loads `configs/baseline.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

# 1) On Colab, connect Google Drive so the project folder is visible.
try:
    from google.colab import drive                      # this import only works on Colab
    drive.mount("/content/drive")
    os.environ.setdefault("DERMALENS_PROJECT_DIR", "/content/drive/MyDrive/DermaLens_AI")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False                                    # local run: set DERMALENS_PROJECT_DIR yourself

# 2) Where the data lives (PROJECT_DIR) and where the code lives (CODE_DIR, normally the same folder).
PROJECT_DIR = Path(os.environ.get("DERMALENS_PROJECT_DIR", Path.cwd().parent))
CODE_DIR = Path(os.environ.get("DERMALENS_CODE_DIR", PROJECT_DIR))

# 3) Colab starts empty every session: install the few libraries it does not ship with.
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")], check=True)

# 4) Make `import dermalens` work.
sys.path.insert(0, str(CODE_DIR / "src"))

from dermalens.config import load_config
from dermalens.paths import ProjectPaths
from dermalens.utils import set_seed

# 5) Folders and settings.
paths = ProjectPaths(PROJECT_DIR).ensure()
CONFIG = load_config(os.environ.get("DERMALENS_CONFIG", CODE_DIR / "configs" / "baseline.yaml"))
set_seed(CONFIG["seed"])
print(paths.describe())

> **Rule:** the test set is used **once**. Everything (temperatures, weights, thresholds) is chosen on validation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dermalens.data import make_loader, resolve_images
from dermalens.utils import get_device, load_json, save_json

info = load_json(paths.splits / "class_info.json")       # classes + weights frozen in notebook 05
CLASSES = info["classes"]                                  # 6 targets + "Other"
WEIGHTS = [info["class_weights"][name] for name in CLASSES]
train = pd.read_csv(paths.splits / "train.csv")
val = pd.read_csv(paths.splits / "val.csv")
IMAGE_ROOT, PATH_COL = resolve_images(train, paths)        # cache if present, else raw files
DEVICE = get_device()
IMAGE_SIZE = CONFIG["data"]["image_size"]
BATCH, WORKERS = CONFIG["train"]["batch_size"], CONFIG["train"]["num_workers"]
USE_META, COLOR = CONFIG["data"]["use_metadata"], CONFIG["data"]["color_constancy"]


def loader(rows, train_mode=False, size=IMAGE_SIZE, use_meta=USE_META, classes=None, label_col="label",
           seed=0):
    """One DataLoader with the project settings (augmentation + metadata dropout only when training)."""
    return make_loader(rows, IMAGE_ROOT, classes or CLASSES, size, train=train_mode, batch_size=BATCH,
                       num_workers=WORKERS, path_col=PATH_COL, label_col=label_col, use_metadata=use_meta,
                       meta_dropout=CONFIG["data"]["meta_dropout"], color_constancy=COLOR, seed=seed)


print(f"train {len(train)} | val {len(val)} | classes {len(CLASSES)} | device {DEVICE} | "
      f"images from {IMAGE_ROOT} ({PATH_COL})")

from dermalens.evaluate import (bootstrap_ci, ensemble_probabilities, expected_calibration_error,
                                fit_temperature, grouped_metrics, headline_metrics, metrics_by_group,
                                per_class_table, plot_confusion_matrix, plot_reliability, predict_logits,
                                softmax)
from dermalens.labels import GROUPED_LABELS, OTHER_CLASS
from dermalens.train import model_from_checkpoint

test = pd.read_csv(paths.splits / "test.csv")
members = load_json(paths.results / "best_models.json")["members"]


def member_logits(member, rows):
    """TTA logits of one ensemble member on a table of rows."""
    model, facts = model_from_checkpoint(paths.root / member["checkpoint"], DEVICE)
    return predict_logits(model, loader(rows, size=facts["image_size"], use_meta=facts["use_metadata"]),
                          DEVICE, tta=True)

## 1. Validation and test logits for each member, with test-time augmentation

TTA averages 4 flipped views of every photo; each member gets its own temperature `T` fitted on
validation so that its probabilities are honest before averaging.

In [ ]:
val_logits, test_logits, temperatures = [], [], []
for member in members:
    v, val_true = member_logits(member, val)
    t, y_true = member_logits(member, test)
    temperatures.append(fit_temperature(v, val_true))
    val_logits.append(v)
    test_logits.append(t)
    single = headline_metrics(val_true, v.argmax(1))
    print(f"{member['model']:<18} T = {temperatures[-1]:.2f}  val acc (TTA) {single['accuracy']:.3f}  "
          f"macro-F1 {single['macro_f1']:.3f}")
val_probs = ensemble_probabilities(val_logits, temperatures)
probs = ensemble_probabilities(test_logits, temperatures)
print("Ensemble val:", {k: round(v, 3) for k, v in headline_metrics(val_true, val_probs.argmax(1)).items()})

## 2. Two views of the test result

* **7-way** - every test photo, including "Other" photos, scored on all 7 outputs.
* **Known classes (6-way)** - only photos of the 6 target classes, the model's answer restricted to
  those 6. This is the number comparable with other papers and with v2.

In [ ]:
other_index = CLASSES.index(OTHER_CLASS) if OTHER_CLASS in CLASSES else None
known_classes = [c for c in CLASSES if c != OTHER_CLASS]
known_rows = np.array([CLASSES[i] != OTHER_CLASS for i in y_true])
known_probs = probs[known_rows][:, [CLASSES.index(c) for c in known_classes]]
known_true = np.array([known_classes.index(CLASSES[i]) for i in y_true[known_rows]])
known_pred = known_probs.argmax(1)
y_pred = probs.argmax(1)
groups_known = test.loc[known_rows, "group_id"].to_numpy()

n_boot = CONFIG["evaluate"]["bootstrap_samples"]
metric_functions = {
    "accuracy": lambda t, p: (t == p).mean(),
    "macro_f1": lambda t, p: headline_metrics(t, p)["macro_f1"],
    "balanced_accuracy": lambda t, p: headline_metrics(t, p)["balanced_accuracy"],
}
headline = {}
for name, function in metric_functions.items():
    point, low, high = bootstrap_ci(known_true, known_pred, groups_known, function, n_boot, CONFIG["seed"])
    headline[name] = {"value": round(point, 4), "ci95": [round(low, 4), round(high, 4)]}
    print(f"6 known classes  {name:<18} {point:.3f}  (95% CI {low:.3f} - {high:.3f})")
seven_way = {k: round(float(v), 4) for k, v in headline_metrics(y_true, y_pred).items()}
print("7-way (with Other):", seven_way)

## 3. Per class and confusion matrix

In [ ]:
class_table = per_class_table(known_true, known_pred, known_classes)
display(class_table)
plot_confusion_matrix(known_true, known_pred, known_classes, paths.figures / "confusion_test.png",
                      title="Test set, ensemble (6 known classes)")
plt.show()

## 4. Grouped taxonomy view

Contact dermatitis is a sub-type of dermatitis, and dermatologists often cannot tell it from atopic
eczema in a photo either. Merging the two into **"Eczema / Dermatitis"** (5 classes) shows how much of
the remaining error is this one clinically-hard pair. Notebook 13 also *trains* on the merged labels.

In [ ]:
grouped = grouped_metrics(known_true, known_pred, known_classes, GROUPED_LABELS)
grouped = {k: round(float(v), 4) for k, v in grouped.items()}
print("Grouped (5 classes, same predictions):", grouped)

## 5. By source and by skin type (fairness)

In [ ]:
results = test.assign(true_label=[CLASSES[i] for i in y_true], predicted=[CLASSES[i] for i in y_pred],
                      confidence=probs.max(1), y_true=y_true, y_pred=y_pred)
results["correct"] = results["y_true"] == results["y_pred"]
fst = results["fst"] if "fst" in results.columns else pd.Series(np.nan, index=results.index)
results["fst_group"] = pd.cut(fst, bins=[0, 2, 4, 6], labels=["FST 1-2", "FST 3-4", "FST 5-6"]).astype("string")
known_results = results[results["true_label"] != OTHER_CLASS]
by_source = metrics_by_group(known_results, "source", CLASSES)
display(by_source)
derma = known_results[known_results["fst_group"].notna()]
by_fst = metrics_by_group(derma, "fst_group", CLASSES)
display(by_fst)
fst_gap = float(by_fst["accuracy"].max() - by_fst["accuracy"].min()) if len(by_fst) >= 2 else float("nan")
print(f"Accuracy gap between skin-type groups: {fst_gap * 100:.1f} points")

## 6. Calibration

In [ ]:
raw = np.mean([softmax(t) for t in test_logits], axis=0)
ece_before, ece_after = expected_calibration_error(raw, y_true), expected_calibration_error(probs, y_true)
print(f"ECE {ece_before:.3f} before -> {ece_after:.3f} after temperature scaling")
plot_reliability(probs, y_true, paths.figures / "reliability_test.png")
plt.show()

## 7. Save (all report numbers come from these files)

In [ ]:
np.save(paths.results / "val_ensemble_probs.npy", val_probs)
np.save(paths.results / "val_true.npy", val_true)
np.save(paths.results / "test_ensemble_probs.npy", probs)
columns = ["image_id", "source", "group_id", "true_label", "predicted", "confidence", "correct", "fst",
           "meta_age", "meta_sex", "meta_regions", PATH_COL]
results[[c for c in columns if c in results.columns]].to_csv(paths.results / "test_predictions.csv", index=False)
save_json({"members": members, "temperatures": temperatures, "headline_known_classes": headline,
           "seven_way": seven_way, "grouped_taxonomy": grouped,
           "per_class": class_table.reset_index().to_dict(orient="records"),
           "by_source": by_source.to_dict(orient="records"), "by_fst": by_fst.to_dict(orient="records"),
           "fst_gap_points": round(fst_gap * 100, 1), "ece_before": ece_before, "ece_after": ece_after,
           "test_images": int(len(test)), "test_images_known": int(known_rows.sum())},
          paths.results / "test_results.json")
print("Saved test_results.json")